# OSEM-ANLM unfolding of an IAEA Compendium spectrum (`unfold_osem_anlm`)

This notebook applies the OSEM-ANLM unfolding method of

> S. Jamaati et al., *Enhanced sparse view CT reconstruction using ordered
> subset expectation maximization and asymptotic non-local means algorithms*,
> Scientific Reports (2026), doi:10.1038/s41598-026-70607-1

adapted to neutron spectrum unfolding.  The method interleaves
ordered-subset expectation maximization (OSEM) updates with the
**two-stage asymptotic non-local means (ANLM)** filter applied to the
intermediate spectrum after every subset update (article pseudo-code
steps 4-5), or once to the final OSEM result (`anlm_mode='post'`).

The ANLM filter implements the article's two-stage scheme:

* stage 1 — NLM with the uniform parameter $h_1 = 0.5\,\sigma$;
* stage 2 — NLM with the point-wise parameter (article eq. 6)
  $$h_2(i) = \Big(\sum_{j\in N_i} w(i,j)^2\,\sigma^2\Big)^{1/2},$$
  i.e. the noise standard deviation smoothed by the stage-1 NLM weights
  $w(i,j)$ computed over the search window $N$ with Gaussian-weighted
  patch distances on the similarity window $\nu$.

The noise level $\sigma$ is either user-supplied (`h`, in log units
when `log_space=True`) or estimated automatically with a robust MAD
estimator on the second differences of the intermediate spectrum
(`estimate_noise_1d`).  For one-dimensional spectra the 2D windows
become index windows on the energy grid, and by default the filter
operates in **log space** (`log_space=True`), where the filtered value
is a weighted geometric mean — scale-free for spectra spanning orders
of magnitude and non-negative by construction.


In [ ]:
# %pip install bssunfold pandas numpy matplotlib


## 1. Detector response functions

We use the built-in GSF response functions (10 Bonner spheres, `0in` -
`18in`, 60 energy bins from 1e-9 to ~631 MeV).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bssunfold import Detector, RF_GSF
from bssunfold.core.unfold_osem import solve_osem
from bssunfold.core.unfold_osem_anlm import anlm_filter_1d, estimate_noise_1d
from bssunfold.utils.comparison import compare_spectra

detector = Detector(RF_GSF)
E = detector.E_MeV
names = detector.detector_names
print(f"Detector grid: {detector.n_energy_bins} bins, "
      f"{E[0]:.1e} - {E[-1]:.1f} MeV")
print("Spheres:", ", ".join(names))
detector.plot_response_functions()


## 2. IAEA Compendium reference spectrum -> detector readings

The compendium CSV stores 61-point Monte-Carlo spectra on its own energy
grid, which differs slightly from the 60-bin detector grid, so the
ground truth is interpolated onto the detector grid *for evaluation
only*.  The readings are the exact foldings of the reference spectrum
through the response functions (no measurement noise is added).


In [ ]:
reference_csv = pd.read_csv(
    '../tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv'
)
print(f"Reference CSV: {len(reference_csv)} rows, "
      f"{len(reference_csv.columns) - 1} spectra, "
      f"{reference_csv['E_MeV'].min():.1e} - "
      f"{reference_csv['E_MeV'].max():.0f} MeV")

SPECTRUM = 't4-14-s.txt_1'   # IAEA Compendium BSA benchmark case

readings = detector.get_effective_readings_for_spectra(
    reference_csv[['E_MeV', SPECTRUM]]
)
print("Effective readings:")
for nm in names:
    print(f"  {nm:>5s}: {readings[nm]:.4g}")

# Ground truth on the detector grid (for evaluation only)
phi_true = np.interp(
    E, reference_csv['E_MeV'].values, reference_csv[SPECTRUM].values
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.loglog(E, phi_true, "k-", lw=1.5)
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title=f"IAEA Compendium spectrum {SPECTRUM} (ground truth)")
ax.grid(True, which="both", ls=":", alpha=0.5)

ax = axes[1]
vals = [readings[nm] for nm in names]
ax.bar(np.arange(len(names)), vals, color="steelblue")
ax.set_yscale("log")
ax.set_xticks(np.arange(len(names)))
ax.set_xticklabels(names, rotation=45)
ax.set(xlabel="sphere", ylabel="reading, a.u.",
       title="Effective Bonner-sphere readings")
ax.grid(True, axis="y", ls=":", alpha=0.5)
fig.tight_layout()
plt.show()


## 3. Unfold with OSEM-ANLM

Two application modes of the ANLM filter are demonstrated:

* **`anlm_mode='subset'`** (default, article pseudo-code) — the filter
  is applied after *every* subset update, suppressing the EM noise as
  soon as it appears;
* **`anlm_mode='post'`** — a single ANLM application to the plain OSEM
  result.

With the automatic noise estimate (`h=None`) the filter is
**non-destructive**: on this noiseless benchmark the EM iterate is
already smooth, the MAD estimator finds little spectrum-level noise and
the filter stays close to the identity — the unfolded spectrum tracks
plain OSEM.  The denoising power of the filter is controlled with the
explicit `h` parameter (section 5).


In [ ]:
result = detector.unfold_osem_anlm(
    readings,
    n_subsets=3,
    max_iterations=100,
    max_neutron_energy=20.0,
    save_result=False,
)

print(f"method            : {result['method']}")
print(f"iterations        : {result['iterations']}  "
      f"(converged: {result['converged']})")
print(f"subsets           : {result['n_subsets']}")
print(f"noise level h     : {result['h']}  (None = automatic estimate)")
print(f"search/sim windows: {result['search_window']}/{result['similarity_window']}")
print(f"residual norm     : {result['residual_norm']:.3e}")

metrics = compare_spectra(
    phi_true, result["spectrum"],
    metrics=["relative_flux_error", "pearson_r", "comprehensive_score"],
)
print("\nquality vs ground truth (subset mode, auto h):")
for key, val in metrics.items():
    print(f"  {key:<30s}: {val:.3f}")

# post mode: single ANLM application to the OSEM result
result_post = detector.unfold_osem_anlm(
    readings,
    n_subsets=3,
    max_iterations=100,
    anlm_mode="post",
    max_neutron_energy=20.0,
    save_result=False,
)
metrics_post = compare_spectra(
    phi_true, result_post["spectrum"],
    metrics=["relative_flux_error", "pearson_r", "comprehensive_score"],
)
print("quality vs ground truth (post mode, auto h):")
for key, val in metrics_post.items():
    print(f"  {key:<30s}: {val:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(E, phi_true, "k-", lw=1.5, label="ground truth")
ax.loglog(E, result["spectrum"], "--", lw=1.5,
          label="OSEM-ANLM subset mode")
ax.loglog(E, result_post["spectrum"], ":", lw=2.0,
          label="OSEM-ANLM post mode")
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="OSEM-ANLM unfolding of the IAEA Compendium spectrum")
ax.legend()
ax.grid(True, which="both", ls=":", alpha=0.5)
fig.tight_layout()
plt.show()


## 4. The ANLM filter in isolation

The 1D filter `anlm_filter_1d` is exported as a standalone building
block.  In the `post` mode the solver output is *exactly* the
composition of plain OSEM with the filter — here we verify that
property and show the automatic noise estimate
(`estimate_noise_1d`) alongside the manually controlled filter
strength.


In [ ]:
mask = E <= 20.0                       # bins kept by the 20 MeV cutoff
A = np.array([
    detector.sensitivities[nm][mask] for nm in names if nm in readings
])
b = np.array([readings[nm] for nm in names if nm in readings])
x0 = np.ones(int(mask.sum()))
x0[0] = 0.0                            # same default as the Detector wrapper
phi_osem, _, _ = solve_osem(A, b, x0, max_iterations=100, n_subsets=3)

h_auto = estimate_noise_1d(phi_osem)   # the estimate used when h=None
phi_filtered = anlm_filter_1d(phi_osem, h=None, log_space=True)

print(f"automatic noise estimate : {h_auto:.3e}  (raw units)")
print(f"max |post mode - (OSEM + ANLM filter)|: "
      f"{np.max(np.abs(result_post['spectrum'][mask] - phi_filtered)):.2e}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(E[mask], phi_true[mask], "k-", lw=1.5, label="ground truth")
ax.loglog(E[mask], phi_osem, lw=1.0, alpha=0.8, label="plain OSEM")
ax.loglog(E[mask], phi_filtered, "--", lw=1.5,
          label="OSEM + ANLM filter (auto h)")
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="ANLM filter applied to the plain OSEM result")
ax.legend()
ax.grid(True, which="both", ls=":", alpha=0.5)
fig.tight_layout()
plt.show()


## 5. The smoothing knob `h`

The article's optimal windows are `search_window=11` (N) and
`similarity_window=3` (nu); the remaining control is the noise level
`sigma` of the two filter stages.  With `log_space=True` it is expressed
in log units, i.e. it acts as a *relative* smoothing scale: `h=0.05`
means the filter averages features smaller than ~5% of the local
fluence.  Increasing `h` monotonically increases the smoothing strength
(and, on this noiseless benchmark, the bias of the reconstruction — the
sharp BSA peak is progressively flattened).


In [ ]:
variants = [
    ("auto h (nearly transparent)", None),
    ("h = 0.02", 0.02),
    ("h = 0.05", 0.05),
    ("h = 0.10", 0.10),
    ("h = 0.30", 0.30),
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(E, phi_true, "k-", lw=1.5, label="ground truth")
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(variants)))
for color, (label, h) in zip(colors, variants):
    res = detector.unfold_osem_anlm(
        readings, n_subsets=3, max_iterations=100, h=h,
        max_neutron_energy=20.0, save_result=False,
    )
    m = compare_spectra(phi_true, res["spectrum"],
                        metrics=["comprehensive_score"])
    ax.loglog(E, res["spectrum"], lw=1.3, color=color,
              label=f"{label}  (score {m['comprehensive_score']:.2f})")
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="Effect of the ANLM filter strength (subset mode)")
ax.legend(fontsize=8)
ax.grid(True, which="both", ls=":", alpha=0.5)
fig.tight_layout()
plt.show()


## 6. Baselines

Two widely used iterative baselines — plain **OSEM** (same subset
schedule, no filtering) and classic **MLEM** — start from the same
initial spectrum for comparison.  On the noiseless benchmark
OSEM-ANLM with the automatic estimate reduces to plain OSEM, which is
the desired non-destructive behaviour; the value of the filter grows
with the noise level of the measurements (per-bin EM noise, inconsistent
readings), exactly the sparse-view regime the article targets.


In [ ]:
osem = detector.unfold_osem(
    readings,
    initial_spectrum=np.full(int((E <= 20.0).sum()), 1.0),
    n_subsets=3,
    max_iterations=100,
    max_neutron_energy=20.0,
    save_result=False,
)
mlem = detector.unfold_mlem(
    readings,
    initial_spectrum=np.full(E.size, 0.5),
    max_iterations=2000,
    save_result=False,
)

labels = ["OSEM-ANLM (subset)", "OSEM-ANLM (post)", "plain OSEM", "MLEM"]
spectra = [result["spectrum"], result_post["spectrum"],
           osem["spectrum"], mlem["spectrum"]]
print(f"{'method':<20s} {'score':>8s} {'pearson_r':>10s} {'rel. flux err':>14s}")
for label, spec in zip(labels, spectra):
    m = compare_spectra(
        phi_true, spec,
        metrics=["comprehensive_score", "pearson_r", "relative_flux_error"],
    )
    print(f"{label:<20s} {m['comprehensive_score']:>8.3f} "
          f"{m['pearson_r']:>10.3f} {m['relative_flux_error']:>14.3f}")


## 7. Summary

* `unfold_osem_anlm` ports the OSEM-ANLM algorithm of Jamaati et al.
  (2026) to neutron spectrum unfolding: ordered-subset EM updates
  interleaved with the two-stage asymptotic non-local means filter
  (stage 1 `h1 = 0.5 sigma`; stage 2 point-wise `h2(i) =
  sqrt(sum_j w(i,j)^2 sigma^2)`, article eq. 6).
* The filter may run after every subset update (`anlm_mode='subset'`,
  article pseudo-code) or once on the OSEM result (`anlm_mode='post'`);
  the standalone `anlm_filter_1d` reproduces the post-mode composition
  exactly.
* The noise level `sigma` is user-supplied (`h`) or estimated
  automatically with a robust MAD estimator; log-space filtering
  (`log_space=True`, default) makes the filter scale-free and
  non-negative by construction.
* On the noiseless IAEA benchmark with 10 spheres the automatic filter
  is non-destructive (tracks plain OSEM); explicit `h` gives monotone
  control of the smoothing strength for noisier measurement campaigns,
  and `calculate_errors=True` provides Monte-Carlo uncertainty
  propagation.
